# NUS ST4253 — Advanced Time-Series Extensions
## From ARIMAX and Kalman filters to GARCH, VAR/VECM, conformal prediction, tree ensembles, RNNs, TCNs, Transformers and foundation models

This notebook is a **second-stage continuation** of a classical ST4253 notebook.

The first notebook established:

$$
\text{stationarity}
\rightarrow
\text{ACF/PACF}
\rightarrow
\text{AR/MA/ARMA}
\rightarrow
\text{ARIMA/ETS}
\rightarrow
\text{diagnostics}
\rightarrow
\text{forecasting}.
$$

This notebook asks:

> What happens when the real world is more complicated than a single stable univariate process?

We progressively introduce:

1. dynamic regression / ARIMAX;
2. intervention analysis;
3. structural breaks;
4. state-space models and Kalman filtering;
5. stochastic volatility;
6. ARCH/GARCH;
7. VAR;
8. cointegration and VECM;
9. hierarchical forecasting and reconciliation;
10. probabilistic forecast scoring;
11. conformal prediction;
12. gradient-boosted trees with lag features;
13. Random Forest forecasting;
14. recurrent neural networks;
15. temporal convolutional networks;
16. Transformers;
17. modern time-series foundation models.

All visualisations use **Bokeh**.

The notebook deliberately combines:
- **controlled simulations** where the truth is known;
- **real datasets** where modelling decisions are less obvious;
- **modular reusable code** instead of repeated ad-hoc cells.

# 0. Learning map

The extensions can be organised into five families.

### A. External information and changing regimes

$$
\boxed{\text{ARIMAX}}
\rightarrow
\boxed{\text{interventions}}
\rightarrow
\boxed{\text{structural breaks}}
$$

### B. Latent-state modelling

$$
\boxed{\text{state-space}}
\rightarrow
\boxed{\text{Kalman filtering}}
\rightarrow
\boxed{\text{stochastic volatility}}
$$

### C. Multivariate dynamics

$$
\boxed{\text{VAR}}
\rightarrow
\boxed{\text{cointegration}}
\rightarrow
\boxed{\text{VECM}}
$$

### D. Forecast systems and uncertainty

$$
\boxed{\text{hierarchies}}
\rightarrow
\boxed{\text{probabilistic scores}}
\rightarrow
\boxed{\text{conformal intervals}}
$$

### E. Machine learning and deep learning

$$
\boxed{\text{lag features}}
\rightarrow
\boxed{\text{trees}}
\rightarrow
\boxed{\text{RNN/TCN/Transformer}}
\rightarrow
\boxed{\text{foundation models}}
$$

In [1]:
import warnings
from dataclasses import dataclass
from abc import ABC, abstractmethod
from typing import Dict, List, Tuple, Optional, Sequence

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.structural import UnobservedComponents
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

from bokeh.io import output_notebook, show
from bokeh.layouts import column, gridplot
from bokeh.models import Band, ColumnDataSource, Span
from bokeh.plotting import figure

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

FAST_MODE = True

output_notebook()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

print("Environment ready.")

Loading BokehJS ...

Environment ready.


# 1. Reusable utilities

A time-series notebook becomes much easier to maintain if we separate:

- visualisation;
- feature engineering;
- model fitting;
- evaluation;
- uncertainty scoring.

The following utilities are deliberately generic.

In [2]:
def time_plot(series_map: Dict[str, pd.Series], title: str, y_label="Value", width=900, height=330):
    p = figure(
        width=width,
        height=height,
        x_axis_type="datetime",
        title=title,
        x_axis_label="Time",
        y_axis_label=y_label,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    dashes = ["solid", "dashed", "dotted", "dotdash", "dashdot"]
    for i, (name, s) in enumerate(series_map.items()):
        s = pd.Series(s).dropna()
        p.line(s.index, s.values, line_width=2, line_dash=dashes[i % len(dashes)], legend_label=name)
    if len(series_map) > 1:
        p.legend.location = "top_left"
        p.legend.click_policy = "hide"
    return p


def sequence_plot(series_map: Dict[str, Sequence[float]], title: str, y_label="Value", width=900, height=320):
    p = figure(
        width=width,
        height=height,
        title=title,
        x_axis_label="Index",
        y_axis_label=y_label,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    dashes = ["solid", "dashed", "dotted", "dotdash", "dashdot"]
    for i, (name, values) in enumerate(series_map.items()):
        arr = np.asarray(values, dtype=float)
        p.line(np.arange(len(arr)), arr, line_width=2, line_dash=dashes[i % len(dashes)], legend_label=name)
    if len(series_map) > 1:
        p.legend.location = "top_left"
        p.legend.click_policy = "hide"
    return p


def metric_table(actual, forecast, model_name):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    rmse = np.sqrt(np.mean((actual - forecast) ** 2))
    mae = np.mean(np.abs(actual - forecast))
    return {"model": model_name, "MAE": mae, "RMSE": rmse}


def lag_matrix(series: pd.Series, lags: Sequence[int], rolling_windows: Sequence[int] = ()) -> pd.DataFrame:
    df = pd.DataFrame({"y": series})
    for lag in lags:
        df[f"lag_{lag}"] = series.shift(lag)
    for w in rolling_windows:
        df[f"roll_mean_{w}"] = series.shift(1).rolling(w).mean()
        df[f"roll_std_{w}"] = series.shift(1).rolling(w).std()

    if isinstance(series.index, pd.DatetimeIndex):
        df["month_sin"] = np.sin(2 * np.pi * series.index.month / 12)
        df["month_cos"] = np.cos(2 * np.pi * series.index.month / 12)

    return df.dropna()


def chronological_xy_split(df: pd.DataFrame, test_size: int):
    train = df.iloc[:-test_size]
    test = df.iloc[-test_size:]
    X_train = train.drop(columns="y")
    y_train = train["y"]
    X_test = test.drop(columns="y")
    y_test = test["y"]
    return X_train, X_test, y_train, y_test


def interval_plot(index, actual, mean, lower, upper, title, y_label="Value"):
    src = ColumnDataSource(pd.DataFrame({
        "date": index,
        "actual": np.asarray(actual),
        "mean": np.asarray(mean),
        "lower": np.asarray(lower),
        "upper": np.asarray(upper),
    }))
    p = figure(
        width=900,
        height=360,
        x_axis_type="datetime",
        title=title,
        x_axis_label="Time",
        y_axis_label=y_label,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p.line("date", "actual", source=src, line_width=2, legend_label="Actual")
    p.line("date", "mean", source=src, line_width=2, line_dash="dashed", legend_label="Forecast")
    p.add_layout(Band(base="date", lower="lower", upper="upper", source=src, fill_alpha=0.18))
    p.legend.location = "top_left"
    return p

# Part I — Dynamic regression / ARIMAX

# 2. Why ordinary regression is not enough

Suppose electricity demand depends on temperature:

$$
Y_t=\beta_0+\beta_1X_t+\varepsilon_t.
$$

Ordinary regression assumes the residual sequence behaves approximately independently.

But in time-series data, residuals often have their own temporal structure:

$$
\varepsilon_t
=
\phi\varepsilon_{t-1}
+
u_t.
$$

Dynamic regression combines:

- regression on external predictors;
- ARIMA-style modelling of the residual process.

A convenient notation is:

$$
Y_t
=
X_t^\top\beta
+
N_t,
$$

where $N_t$ follows ARIMA.

This family is often called **ARIMAX**, **regression with ARIMA errors**, or dynamic regression.

In [3]:
n = 300
dates = pd.date_range("2000-01-01", periods=n, freq="MS")

x = 10 + 3 * np.sin(2 * np.pi * np.arange(n) / 12) + rng.normal(0, 0.7, n)

err = np.zeros(n)
innov = rng.normal(0, 1.0, n)
for t in range(1, n):
    err[t] = 0.75 * err[t-1] + innov[t]

y = 20 + 1.8 * x + err

dyn_df = pd.DataFrame({"y": y, "x": x}, index=dates)

show(column(
    time_plot({"Target y": dyn_df["y"]}, "Dynamic regression target"),
    time_plot({"External regressor x": dyn_df["x"]}, "External driver"),
))

## Two competing models

We compare:

1. ordinary least squares;
2. regression with AR(1) errors via `SARIMAX(..., exog=...)`.

The regression coefficients may look similar, but the second model explicitly recognises that residuals are serially correlated.

In [4]:
train = dyn_df.iloc[:-48]
test = dyn_df.iloc[-48:]

X_ols = sm.add_constant(train[["x"]])
ols = sm.OLS(train["y"], X_ols).fit()
ols_pred = ols.predict(sm.add_constant(test[["x"]], has_constant="add"))

arimax = SARIMAX(
    train["y"],
    exog=train[["x"]],
    order=(1, 0, 0),
    trend="c",
    enforce_stationarity=False,
).fit(disp=False)

arimax_pred = arimax.get_forecast(
    steps=len(test),
    exog=test[["x"]],
).predicted_mean

display(pd.DataFrame([
    metric_table(test["y"], ols_pred, "OLS"),
    metric_table(test["y"], arimax_pred, "ARIMAX / dynamic regression"),
]).round(4))

,model,MAE,RMSE
0,OLS,1.1256,1.3488
1,ARIMAX / dynamic regression,1.0978,1.3217


In [5]:
show(time_plot(
    {
        "Actual": test["y"],
        "OLS": pd.Series(ols_pred, index=test.index),
        "ARIMAX": pd.Series(arimax_pred, index=test.index),
    },
    "OLS versus dynamic regression",
))

# 3. Dynamic-regression interpretation

Dynamic regression separates two ideas:

$$
\boxed{\text{systematic external effect}}
+
\boxed{\text{serially correlated unexplained dynamics}}
$$

This is useful when forecasting depends on:

- weather;
- promotions;
- holidays;
- policy variables;
- interest rates;
- prices;
- interventions.

A crucial practical caveat:

> To forecast $Y_{t+h}$ using future $X_{t+h}$, the future regressor must itself be known or forecast.

So ARIMAX does not make the exogenous-variable problem disappear.

# Part II — Intervention analysis

# 4. What is an intervention?

An intervention is a known event that changes the observed process.

Examples:

- policy implementation;
- advertising campaign;
- system migration;
- pandemic lockdown;
- price change;
- machine repair.

Interventions can be encoded as regressors.

### Pulse intervention

Only one period changes:

$$
I_t=
\begin{cases}
1,& t=T_0\\
0,& \text{otherwise}
\end{cases}
$$

### Step intervention

A persistent level shift:

$$
I_t=
\begin{cases}
0,& t<T_0\\
1,& t\ge T_0
\end{cases}
$$

In [44]:
n = 240
dates = pd.date_range("2005-01-01", periods=n, freq="MS")
t = np.arange(n)

intervention_start = 140
step = (t >= intervention_start).astype(int)

noise = np.zeros(n)
eps = rng.normal(0, 1.0, n)
for i in range(1, n):
    noise[i] = 0.5 * noise[i-1] + eps[i]

y_int = 50 + 0.05*t + 8*step + noise
intervention_df = pd.DataFrame({"y": y_int, "step": step}, index=dates)

p = time_plot({"Observed": intervention_df["y"]}, "Synthetic intervention: persistent level shift")
p.add_layout(Span(location=dates[intervention_start].timestamp()*1000, dimension="height", line_dash="dashed"))
show(p)

In [45]:
model_no_int = SARIMAX(
    intervention_df["y"],
    order=(1,0,0),
    trend="ct",
    enforce_stationarity=False,
).fit(disp=False)

model_with_int = SARIMAX(
    intervention_df["y"],
    exog=intervention_df[["step"]],
    order=(1,0,0),
    trend="ct",
    enforce_stationarity=False,
).fit(disp=False)

comparison = pd.DataFrame({
    "model": ["Without intervention", "With intervention"],
    "AIC": [model_no_int.aic, model_with_int.aic],
    "BIC": [model_no_int.bic, model_with_int.bic],
})

display(comparison.round(3))
print("Estimated intervention coefficient:")
print(model_with_int.params.filter(like="step"))

,model,AIC,BIC
0,Without intervention,794.561,808.467
1,With intervention,700.319,717.701


Estimated intervention coefficient:
step    7.598711
dtype: float64


## Interpretation

The step coefficient estimates the persistent level shift **after accounting for trend and serial correlation**.

This is much more defensible than simply comparing pre/post means when the observations are autocorrelated.

Intervention analysis is essentially:

$$
\boxed{\text{causal event encoding}}
+
\boxed{\text{time-series error model}}
$$

but causal interpretation still requires the event timing and counterfactual assumptions to be credible.

# Part III — Structural breaks

# 5. Intervention versus structural break

An intervention is usually **known externally**.

A structural break is a change in the data-generating relationship itself.

Examples:

- mean changes;
- trend changes;
- regression coefficient changes;
- volatility changes.

Suppose

$$
Y_t=
\begin{cases}
\beta_0+\beta_1t+\varepsilon_t,& t<T_b\\
\gamma_0+\gamma_1t+\varepsilon_t,& t\ge T_b.
\end{cases}
$$

A single global model may average across two incompatible regimes.

In [46]:
n = 260
t = np.arange(n)
break_idx = 150

y_break = np.where(
    t < break_idx,
    20 + 0.03*t,
    10 + 0.10*t
) + rng.normal(0, 1.2, n)

dates = pd.date_range("2000-01-01", periods=n, freq="MS")
break_series = pd.Series(y_break, index=dates)

show(time_plot({"Series": break_series}, "Synthetic structural break in level/trend"))

# 6. Searching for a candidate break point

For teaching purposes, we can fit a piecewise linear regression at each candidate split and choose the split with the smallest total residual sum of squares.

This is conceptually similar to changepoint detection.

It is intentionally simple; specialised packages such as `ruptures` provide more sophisticated algorithms for multiple changes.

In [47]:
def piecewise_break_search(y: pd.Series, min_segment=36) -> pd.DataFrame:
    arr = np.asarray(y, dtype=float)
    t = np.arange(len(arr))
    rows = []

    for b in range(min_segment, len(arr)-min_segment):
        left_X = sm.add_constant(t[:b])
        right_X = sm.add_constant(t[b:])
        left_fit = sm.OLS(arr[:b], left_X).fit()
        right_fit = sm.OLS(arr[b:], right_X).fit()
        ssr = np.sum(left_fit.resid**2) + np.sum(right_fit.resid**2)
        rows.append({"break_index": b, "SSR": ssr})

    return pd.DataFrame(rows)

break_scan = piecewise_break_search(break_series)
best_break = int(break_scan.loc[break_scan["SSR"].idxmin(), "break_index"])

display(break_scan.nsmallest(5, "SSR"))
print("Estimated break date:", break_series.index[best_break])

,break_index,SSR
107,143,417.469104
108,144,417.815795
106,142,418.039551
105,141,418.943031
92,128,420.037201


Estimated break date: 2011-12-01 00:00:00


In [48]:
p = figure(
    width=900, height=320,
    title="Candidate break-point objective",
    x_axis_label="Candidate break index",
    y_axis_label="Piecewise SSR",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
p.line(break_scan["break_index"], break_scan["SSR"], line_width=2)
p.add_layout(Span(location=best_break, dimension="height", line_dash="dashed"))
show(p)

# Part IV — State-space models and Kalman filtering

# 7. The state-space idea

Many time-series processes are easier to describe using **latent states**.

A generic linear Gaussian state-space model is:

### Observation equation

$$
y_t=Z_t\alpha_t+\varepsilon_t
$$

### State transition equation

$$
\alpha_t=T_t\alpha_{t-1}+R_t\eta_t.
$$

The state $\alpha_t$ is not directly observed.

Examples:

- local level;
- trend;
- seasonal components;
- latent volatility;
- latent economic cycle.

The **Kalman filter** recursively estimates the hidden state as new observations arrive.

# 8. Kalman filter: predict → update

At each time step:

### Prediction

$$
\hat\alpha_{t|t-1}
=
T\hat\alpha_{t-1|t-1}
$$

### Innovation

$$
v_t=y_t-Z\hat\alpha_{t|t-1}
$$

### Kalman gain

$$
K_t=P_{t|t-1}Z^\top
\left(
ZP_{t|t-1}Z^\top+H
\right)^{-1}
$$

### Update

$$
\hat\alpha_{t|t}
=
\hat\alpha_{t|t-1}+K_tv_t.
$$

Intuitively:

> prior state estimate + reliability-weighted new information.

If measurement noise is large, the filter trusts the state model more.  
If state uncertainty is large, it reacts more strongly to the new observation.

In [49]:
nile = sm.datasets.nile.load_pandas().data.copy()
nile.index = pd.to_datetime(nile["year"].astype(int).astype(str) + "-01-01")
nile_series = nile["volume"].astype(float)

ucm = UnobservedComponents(
    nile_series,
    level="local linear trend",
)
ucm_fit = ucm.fit(disp=False)

smoothed_level = pd.Series(
    ucm_fit.level.smoothed,
    index=nile_series.index,
)
smoothed_trend = pd.Series(
    ucm_fit.trend.smoothed,
    index=nile_series.index,
)

show(column(
    time_plot(
        {"Observed Nile flow": nile_series, "Smoothed latent level": smoothed_level},
        "State-space local linear trend model",
        y_label="Annual Nile flow",
    ),
    time_plot(
        {"Latent trend": smoothed_trend},
        "Estimated latent trend state",
        y_label="Trend",
    )
))

## Why this is powerful

ARIMA primarily describes autocorrelation in observed/differenced values.

State-space modelling asks a different question:

> Which **unobserved components** evolve through time and jointly generate what we observe?

This makes the framework natural for:

- irregular observations;
- missing values;
- latent trends;
- time-varying parameters;
- signal extraction;
- online filtering.

# Part V — Stochastic volatility

# 9. Why constant variance is often unrealistic

Financial returns may have mean close to zero while volatility changes dramatically through time.

A stochastic-volatility model introduces an unobserved log-variance:

$$
r_t=\exp(h_t/2)\varepsilon_t
$$

with

$$
h_t=\mu+\phi(h_{t-1}-\mu)+\eta_t.
$$

Now volatility itself is a latent stochastic process.

This differs from GARCH, where conditional variance is a deterministic recursion of past squared shocks and past variances.

In [50]:
n = 700
mu = -0.7
phi = 0.97
sigma_eta = 0.18

h = np.zeros(n)
h[0] = mu
eta = rng.normal(0, sigma_eta, n)
eps = rng.normal(0, 1, n)

for t in range(1, n):
    h[t] = mu + phi*(h[t-1]-mu) + eta[t]

sv_returns = np.exp(h/2) * eps
sv_sigma = np.exp(h/2)

show(column(
    sequence_plot({"Returns": sv_returns}, "Stochastic-volatility returns", y_label="Return"),
    sequence_plot({"Latent sigma_t": sv_sigma}, "Latent time-varying volatility", y_label="Conditional sigma"),
))

# Part VI — ARCH/GARCH

# 10. ARCH intuition

ARCH models say that large recent shocks increase current conditional variance.

For ARCH(1):

$$
\sigma_t^2
=
\omega+\alpha\varepsilon_{t-1}^2.
$$

GARCH adds persistence in variance itself:

$$
\sigma_t^2
=
\omega
+
\alpha\varepsilon_{t-1}^2
+
\beta\sigma_{t-1}^2.
$$

For covariance stationarity of the variance process, a common requirement is:

$$
\alpha+\beta<1.
$$

When $\alpha+\beta$ is close to one, volatility shocks decay slowly.

# 11. Educational GARCH(1,1) maximum likelihood implementation

The specialist `arch` package is not installed in this runtime, so this notebook implements a compact Gaussian GARCH(1,1) estimator using `scipy.optimize`.

This is useful pedagogically because every recursion is visible.

For production work, prefer a specialist package such as `arch`, which provides richer diagnostics, distributions and robust inference.

In [51]:
@dataclass
class Garch11Result:
    omega: float
    alpha: float
    beta: float
    sigma2: np.ndarray
    success: bool
    fun: float


class GaussianGARCH11:
    def fit(self, returns: Sequence[float]) -> Garch11Result:
        r = np.asarray(returns, dtype=float)
        r = r - np.mean(r)
        var0 = np.var(r)

        def unpack(theta):
            # Stable parameterisation guaranteeing positive parameters
            omega = np.exp(theta[0])
            a_raw = np.exp(theta[1])
            b_raw = np.exp(theta[2])
            denom = 1 + a_raw + b_raw
            alpha = 0.999 * a_raw / denom
            beta = 0.999 * b_raw / denom
            return omega, alpha, beta

        def objective(theta):
            omega, alpha, beta = unpack(theta)
            sigma2 = np.empty_like(r)
            sigma2[0] = max(var0, 1e-8)

            for t in range(1, len(r)):
                sigma2[t] = omega + alpha*r[t-1]**2 + beta*sigma2[t-1]
                sigma2[t] = max(sigma2[t], 1e-10)

            nll = 0.5*np.sum(np.log(2*np.pi) + np.log(sigma2) + r**2/sigma2)
            return nll

        x0 = np.log([0.01*var0 + 1e-8, 0.08, 0.88])
        opt = minimize(objective, x0=x0, method="Nelder-Mead", options={"maxiter": 4000})

        omega, alpha, beta = unpack(opt.x)
        sigma2 = np.empty_like(r)
        sigma2[0] = max(var0, 1e-8)
        for t in range(1, len(r)):
            sigma2[t] = omega + alpha*r[t-1]**2 + beta*sigma2[t-1]

        return Garch11Result(
            omega=omega,
            alpha=alpha,
            beta=beta,
            sigma2=sigma2,
            success=opt.success,
            fun=opt.fun,
        )

In [52]:
# Simulate a GARCH(1,1) process
n = 1200
omega_true, alpha_true, beta_true = 0.05, 0.08, 0.90

garch_r = np.zeros(n)
garch_var = np.zeros(n)
garch_var[0] = omega_true / (1-alpha_true-beta_true)

z = rng.normal(size=n)

for t in range(1, n):
    garch_var[t] = (
        omega_true
        + alpha_true*garch_r[t-1]**2
        + beta_true*garch_var[t-1]
    )
    garch_r[t] = np.sqrt(garch_var[t]) * z[t]

garch_fit = GaussianGARCH11().fit(garch_r)

display(pd.DataFrame({
    "parameter": ["omega", "alpha", "beta", "alpha+beta"],
    "true": [omega_true, alpha_true, beta_true, alpha_true+beta_true],
    "estimated": [
        garch_fit.omega,
        garch_fit.alpha,
        garch_fit.beta,
        garch_fit.alpha + garch_fit.beta,
    ],
}).round(4))

,parameter,true,estimated
0,omega,0.05,0.0506
1,alpha,0.08,0.0662
2,beta,0.90,0.9081
3,alpha+beta,0.98,0.9743


In [53]:
show(column(
    sequence_plot({"Returns": garch_r}, "GARCH returns"),
    sequence_plot(
        {
            "True sigma": np.sqrt(garch_var),
            "Estimated sigma": np.sqrt(garch_fit.sigma2),
        },
        "True versus estimated conditional volatility",
        y_label="Sigma",
    ),
))

# 12. ARCH/GARCH versus stochastic volatility

### GARCH

Variance is recursively determined by observed information:

$$
\sigma_t^2
=
\omega+\alpha r_{t-1}^2+\beta\sigma_{t-1}^2.
$$

### Stochastic volatility

Log-variance has its own random innovation:

$$
h_t=\mu+\phi(h_{t-1}-\mu)+\eta_t.
$$

So the volatility state itself is random and latent.

This makes stochastic-volatility models naturally state-space/Bayesian, while GARCH is often easier to fit by direct likelihood.

# Part VII — VAR

# 13. Why univariate models can be inadequate

Suppose GDP growth, consumption growth and investment growth interact.

A univariate AR model for GDP ignores information in the other series.

A Vector Autoregression treats every variable as potentially influenced by lagged values of every variable.

For VAR($p$):

$$
\mathbf y_t
=
\mathbf c
+
A_1\mathbf y_{t-1}
+\cdots+
A_p\mathbf y_{t-p}
+
\boldsymbol\varepsilon_t.
$$

For three variables, $\mathbf y_t$ is a $3\times1$ vector and each $A_i$ is a $3\times3$ coefficient matrix.

In [54]:
macro = sm.datasets.macrodata.load_pandas().data.copy()

macro["date"] = pd.PeriodIndex(
    year=macro["year"].astype(int),
    quarter=macro["quarter"].astype(int),
    freq="Q",
).to_timestamp()

macro = macro.set_index("date")

var_data = pd.DataFrame({
    "gdp_growth": 100*np.log(macro["realgdp"]).diff(),
    "cons_growth": 100*np.log(macro["realcons"]).diff(),
    "inv_growth": 100*np.log(macro["realinv"]).diff(),
}).dropna()

show(column(
    time_plot({"GDP growth": var_data["gdp_growth"]}, "Quarterly real GDP growth"),
    time_plot({"Consumption growth": var_data["cons_growth"]}, "Quarterly real consumption growth"),
    time_plot({"Investment growth": var_data["inv_growth"]}, "Quarterly real investment growth"),
))

In [55]:
var_model = VAR(var_data)
lag_selection = var_model.select_order(maxlags=8)

display(lag_selection.summary())

selected_lag = int(lag_selection.aic)
selected_lag = max(1, selected_lag)
print("AIC-selected lag:", selected_lag)

var_fit = var_model.fit(selected_lag)
print(var_fit.summary())

,AIC,BIC,FPE,HQIC
0,-0.08408,-0.03355,0.9194,-0.06362
1,-0.3953*,-0.1932*,0.6735*,-0.3134*
2,-0.3843,-0.03052,0.6810,-0.2410
3,-0.3817,0.1237,0.6829,-0.1770
4,-0.3789,0.2780,0.6850,-0.1129
5,-0.3642,0.4444,0.6956,-0.03677
6,-0.3265,0.6336,0.7228,0.06230
7,-0.3031,0.8086,0.7407,0.1470
8,-0.2953,0.9680,0.7475,0.2162


AIC-selected lag: 1
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Wed, 09, Sep, 2026
Time:                     16:53:24
--------------------------------------------------------------------
No. of Equations:         3.00000    BIC:                  -0.107734
Nobs:                     201.000    HQIC:                 -0.225146
Log likelihood:          -812.973    FPE:                   0.737174
AIC:                    -0.304946    Det(Omega_mle):        0.694859
--------------------------------------------------------------------
Results for equation gdp_growth
                    coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------------
const                  0.357952         0.091137            3.928           0.000
L1.gdp_growth         -0.338056         0.172084           -1.964           0.049
L1.cons_growth         0.746283

# 14. Impulse-response functions

An impulse-response function asks:

> If one variable receives a one-time shock now, how does the entire system respond over future periods?

For example:

$$
\text{shock to investment growth}
\rightarrow
\text{future GDP/consumption/investment responses}.
$$

This is a dynamic-system concept, not merely a forecasting metric.

In [18]:
irf = var_fit.irf(12)
responses = irf.irfs

names = list(var_data.columns)
shock_idx = names.index("inv_growth")

p = figure(
    width=900, height=350,
    title="Impulse responses to a shock in investment growth",
    x_axis_label="Horizon (quarters)",
    y_axis_label="Response",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

dashes = ["solid", "dashed", "dotted"]
for response_idx, name in enumerate(names):
    p.line(
        np.arange(responses.shape[0]),
        responses[:, response_idx, shock_idx],
        line_width=2,
        line_dash=dashes[response_idx],
        legend_label=name,
    )

p.add_layout(Span(location=0, dimension="width", line_width=1))
p.legend.location = "top_right"
show(p)

# Part VIII — Cointegration and VECM

# 15. Why differencing can lose long-run information

Two series may each be non-stationary but move together over the long run.

Suppose:

$$
x_t=x_{t-1}+u_t
$$

and

$$
y_t=1.5x_t+z_t
$$

where $z_t$ is stationary.

Then both $x_t$ and $y_t$ are non-stationary, but

$$
y_t-1.5x_t=z_t
$$

is stationary.

The pair is **cointegrated**.

Cointegration means that a stable long-run equilibrium relationship exists even though individual levels wander.

In [56]:
n = 450
u = rng.normal(0, 1, n)
x = np.cumsum(u)

z = np.zeros(n)
e = rng.normal(0, 0.8, n)
for t in range(1, n):
    z[t] = 0.65*z[t-1] + e[t]

y = 1.5*x + z

dates = pd.date_range("2000-01-01", periods=n, freq="MS")
coint_df = pd.DataFrame({"x": x, "y": y}, index=dates)

show(time_plot({"x": coint_df["x"], "y": coint_df["y"]}, "Synthetic cointegrated pair"))

In [57]:
eg_stat, eg_pvalue, eg_crit = coint(coint_df["y"], coint_df["x"])

print("Engle-Granger statistic:", round(eg_stat, 4))
print("p-value:", round(eg_pvalue, 6))
print("critical values:", eg_crit)

spread = coint_df["y"] - 1.5*coint_df["x"]

display(pd.DataFrame([
    {"series": "x", "ADF_p": adfuller(coint_df["x"])[1]},
    {"series": "y", "ADF_p": adfuller(coint_df["y"])[1]},
    {"series": "y - 1.5x", "ADF_p": adfuller(spread)[1]},
]).round(6))

Engle-Granger statistic: -8.0165
p-value: 0.0
critical values: [-3.92099806 -3.34977208 -3.05390937]


,series,ADF_p
0,x,0.065986
1,y,0.013427
2,y - 1.5x,0.000000


In [21]:
show(time_plot({"Cointegrating spread": spread}, "Stationary long-run equilibrium error"))

# 16. Error-correction mechanism

If the system moves away from long-run equilibrium, future changes may pull it back.

A two-variable VECM has the form

$$
\Delta\mathbf y_t
=
\Pi\mathbf y_{t-1}
+
\Gamma_1\Delta\mathbf y_{t-1}
+\cdots
+
\varepsilon_t.
$$

The matrix

$$
\Pi=\alpha\beta^\top
$$

contains:

- $\beta$: long-run cointegrating relationships;
- $\alpha$: speeds of adjustment back toward equilibrium.

This gives VECM a useful interpretation:

$$
\boxed{\text{short-run changes}}
+
\boxed{\text{long-run equilibrium correction}}.
$$

In [58]:
vecm = VECM(
    coint_df[["x", "y"]],
    k_ar_diff=1,
    coint_rank=1,
    deterministic="co",
)
vecm_fit = vecm.fit()

print(vecm_fit.summary())

Det. terms outside the coint. relation & lagged endog. parameters for equation x
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0617      0.051     -1.208      0.227      -0.162       0.038
L1.x           0.0240      0.106      0.225      0.822      -0.185       0.233
L1.y          -0.0156      0.062     -0.252      0.801      -0.137       0.105
Det. terms outside the coint. relation & lagged endog. parameters for equation y
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1374      0.088     -1.569      0.117      -0.309       0.034
L1.x           0.1621      0.182      0.889      0.374      -0.195       0.520
L1.y          -0.0949      0.106     -0.897      0.370      -0.302       0.113
                 Loading coefficients (alpha) fo

# Part IX — Hierarchical forecasting

# 17. Forecasts often need to add up

Suppose total sales are:

$$
T_t=N_t+S_t.
$$

If we forecast each series independently, we may obtain:

$$
\hat T_t\neq \hat N_t+\hat S_t.
$$

That is **incoherent**.

Hierarchical forecasting imposes aggregation constraints.

In [23]:
n = 180
dates = pd.date_range("2010-01-01", periods=n, freq="MS")
t = np.arange(n)

north = 100 + 0.35*t + 8*np.sin(2*np.pi*t/12) + rng.normal(0, 4, n)
south = 80 + 0.22*t + 6*np.sin(2*np.pi*(t+2)/12) + rng.normal(0, 3.5, n)
total = north + south

hier = pd.DataFrame({"Total": total, "North": north, "South": south}, index=dates)

show(time_plot(
    {"Total": hier["Total"], "North": hier["North"], "South": hier["South"]},
    "Simple forecast hierarchy",
))

# 18. Base forecasts and reconciliation

Represent the hierarchy using a summing matrix:

$$
\mathbf y_t=S\mathbf b_t
$$

with bottom-level vector

$$
\mathbf b_t=
\begin{bmatrix}
N_t\\
S_t
\end{bmatrix}
$$

and

$$
S=
\begin{bmatrix}
1&1\\
1&0\\
0&1
\end{bmatrix}.
$$

We demonstrate:

- independent base forecasts;
- bottom-up reconciliation;
- OLS projection reconciliation.

In [24]:
train_h = hier.iloc[:-24]
test_h = hier.iloc[-24:]

def simple_trend_forecast(s: pd.Series, horizon: int) -> np.ndarray:
    t = np.arange(len(s))
    X = sm.add_constant(t)
    fit = sm.OLS(s.values, X).fit()
    future_t = np.arange(len(s), len(s)+horizon)
    return fit.predict(sm.add_constant(future_t))

base_total = simple_trend_forecast(train_h["Total"], 24)
base_north = simple_trend_forecast(train_h["North"], 24)
base_south = simple_trend_forecast(train_h["South"], 24)

base = np.column_stack([base_total, base_north, base_south])

S = np.array([
    [1, 1],
    [1, 0],
    [0, 1],
], dtype=float)

# Bottom-up: discard total base forecast and aggregate bottom series
bottom = np.column_stack([base_north, base_south])
bottom_up = (S @ bottom.T).T

# OLS reconciliation: project all base forecasts onto coherent subspace
P = S @ np.linalg.inv(S.T @ S) @ S.T
ols_reconciled = (P @ base.T).T

coherence = pd.DataFrame({
    "base_error": base[:,0] - (base[:,1] + base[:,2]),
    "bottom_up_error": bottom_up[:,0] - (bottom_up[:,1] + bottom_up[:,2]),
    "ols_reconciled_error": ols_reconciled[:,0] - (ols_reconciled[:,1] + ols_reconciled[:,2]),
}, index=test_h.index)

display(coherence.abs().mean().to_frame("mean_abs_coherence_error"))

,mean_abs_coherence_error
base_error,1.018445e-13
bottom_up_error,0.000000e+00
ols_reconciled_error,1.657933e-14


In [59]:
show(time_plot(
    {
        "Actual total": test_h["Total"],
        "Independent total forecast": pd.Series(base[:,0], index=test_h.index),
        "Bottom-up total": pd.Series(bottom_up[:,0], index=test_h.index),
        "OLS-reconciled total": pd.Series(ols_reconciled[:,0], index=test_h.index),
    },
    "Hierarchical forecast reconciliation",
))

# Part X — Probabilistic forecast scoring

# 19. Point forecasts are not enough

A probabilistic forecast specifies a distribution:

$$
F_t(y)=P(Y_t\le y\mid\mathcal F_{t-1}).
$$

A good distribution should be:

- **calibrated**: observed frequencies match predicted probabilities;
- **sharp**: uncertainty intervals are not unnecessarily wide.

Proper scoring rules reward honest probabilistic forecasts.

# 20. Pinball / quantile loss

For quantile level $\tau$:

$$
L_\tau(y,q)
=
\begin{cases}
\tau(y-q),&y\ge q\\
(1-\tau)(q-y),&y<q.
\end{cases}
$$

This asymmetric loss makes the optimal prediction the $\tau$-quantile.

It is fundamental for direct quantile forecasting.

In [60]:
def pinball_loss(y, q, tau):
    y = np.asarray(y)
    q = np.asarray(q)
    err = y - q
    return np.mean(np.maximum(tau*err, (tau-1)*err))


def interval_score(y, lower, upper, alpha=0.1):
    y = np.asarray(y)
    lower = np.asarray(lower)
    upper = np.asarray(upper)
    width = upper - lower
    below = (lower - y) * (y < lower)
    above = (y - upper) * (y > upper)
    return np.mean(width + (2/alpha)*below + (2/alpha)*above)


def gaussian_crps(y, mu, sigma):
    y = np.asarray(y, dtype=float)
    mu = np.asarray(mu, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    z = (y - mu) / sigma
    return np.mean(
        sigma * (
            z * (2*norm.cdf(z)-1)
            + 2*norm.pdf(z)
            - 1/np.sqrt(np.pi)
        )
    )

In [61]:
n = 500
truth = rng.normal(0, 1, n)

models_prob = {
    "Well-calibrated": (np.zeros(n), np.ones(n)),
    "Overconfident": (np.zeros(n), np.full(n, 0.45)),
    "Underconfident": (np.zeros(n), np.full(n, 1.8)),
    "Biased": (np.full(n, 0.6), np.ones(n)),
}

rows = []

for name, (mu, sigma) in models_prob.items():
    z90 = norm.ppf(0.95)
    lower = mu - z90*sigma
    upper = mu + z90*sigma
    coverage = np.mean((truth >= lower) & (truth <= upper))

    rows.append({
        "model": name,
        "CRPS": gaussian_crps(truth, mu, sigma),
        "90%_coverage": coverage,
        "mean_interval_width": np.mean(upper-lower),
        "interval_score": interval_score(truth, lower, upper, alpha=0.10),
    })

display(pd.DataFrame(rows).sort_values("CRPS").round(4))

,model,CRPS,90%_coverage,mean_interval_width,interval_score
0,Well-calibrated,0.5913,0.862,3.2897,4.3609
2,Underconfident,0.6468,0.996,5.9215,5.9305
1,Overconfident,0.6513,0.552,1.4804,7.4784
3,Biased,0.6941,0.802,3.2897,5.4494


## Calibration versus sharpness

An extremely wide interval can achieve excellent empirical coverage but be useless.

An extremely narrow interval may be sharp but badly under-cover.

A proper probabilistic score balances both.

This is why **coverage alone is insufficient**.

# Part XI — Conformal prediction for time series

# 21. Distribution-free predictive intervals

Conformal prediction builds intervals from empirical residual/nonconformity scores rather than assuming a particular error distribution.

In ordinary split conformal regression:

1. fit a model on training data;
2. compute absolute residuals on calibration data;
3. take an upper quantile $q$;
4. predict

$$
[\hat y-q,\hat y+q].
$$

For time series, exchangeability is generally violated, so time-aware adaptations are preferable.

Here we use a **rolling residual calibration window**, which is a simple practical bridge toward adaptive conformal forecasting.

In [62]:
# Use the CO2 monthly series again.
co2 = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate("time")

feat = lag_matrix(
    co2,
    lags=[1,2,3,6,12,13,24],
    rolling_windows=[3,6,12],
)

cal_size = 60
test_size = 48

train_part = feat.iloc[:-(cal_size+test_size)]
cal_part = feat.iloc[-(cal_size+test_size):-test_size]
test_part = feat.iloc[-test_size:]

Xtr, ytr = train_part.drop(columns="y"), train_part["y"]
Xcal, ycal = cal_part.drop(columns="y"), cal_part["y"]
Xte, yte = test_part.drop(columns="y"), test_part["y"]

base_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    random_state=SEED,
).fit(Xtr, ytr)

cal_pred = base_model.predict(Xcal)
scores = np.abs(ycal.values - cal_pred)

alpha = 0.10
q = np.quantile(scores, 1-alpha, method="higher")

test_pred = base_model.predict(Xte)
lower = test_pred - q
upper = test_pred + q

coverage = np.mean((yte.values >= lower) & (yte.values <= upper))

print("Calibration quantile q =", round(float(q), 4))
print("Empirical test coverage =", round(float(coverage), 4))

Calibration quantile q = 7.4061
Empirical test coverage = 0.0208


In [63]:
show(interval_plot(
    yte.index,
    yte,
    test_pred,
    lower,
    upper,
    "Time-ordered split conformal interval (90% nominal)",
    y_label="CO2",
))

## Important caveat

Classical conformal guarantees rely on exchangeability.

Time series exhibit dependence and drift, so stronger methods include:

- rolling-window conformal;
- adaptive conformal inference;
- conformalized quantile regression;
- block conformal methods;
- weighted conformal under covariate shift.

The core lesson is still valuable:

> Forecast intervals can be calibrated using empirical forecast errors rather than relying exclusively on a Gaussian likelihood.

# Part XII — Tree-based machine-learning forecasting

# 22. Convert forecasting into supervised learning

Tree models do not understand temporal order automatically.

We create features such as:

$$
Y_{t-1},Y_{t-2},Y_{t-12},
$$

rolling summaries,

and calendar encodings.

Then learn:

$$
Y_t=f(X_t).
$$

This is sometimes called the **lag-feature approach**.

In [30]:
ml_df = lag_matrix(
    co2,
    lags=[1,2,3,4,5,6,12,13,18,24],
    rolling_windows=[3,6,12,24],
)

X_train, X_test, y_train, y_test = chronological_xy_split(ml_df, test_size=60)

display(X_train.head())
print(X_train.shape, X_test.shape)

,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_12,lag_13,lag_18,lag_24,roll_mean_3,roll_std_3,roll_mean_6,roll_std_6,roll_mean_12,roll_std_12,roll_mean_24,roll_std_24,month_sin,month_cos
1960-03-01,316.975,316.380,315.525,314.875,313.400,313.825,316.733333,316.700000,313.500000,316.100000,316.293333,0.728875,315.163333,1.405477,316.096944,1.603955,315.761422,1.529488,1.000000e+00,6.123234e-17
1960-04-01,317.575,316.975,316.380,315.525,314.875,313.400,317.675000,316.733333,313.463115,317.200000,316.976667,0.597502,315.788333,1.520400,316.167083,1.651997,315.822880,1.572708,8.660254e-01,-5.000000e-01
1960-05-01,319.120,317.575,316.975,316.380,315.525,314.875,318.325000,317.675000,313.425000,317.433333,317.890000,1.106650,316.741667,1.516633,316.287500,1.816388,315.902880,1.690245,5.000000e-01,-8.660254e-01
1960-06-01,319.925,319.120,317.575,316.975,316.380,315.525,318.025000,318.325000,314.700000,316.514344,318.873333,1.194261,317.583333,1.667311,316.420833,2.026161,316.006700,1.856666,1.224647e-16,-1.000000e+00
1960-07-01,319.450,319.925,319.120,317.575,316.975,316.380,316.525000,318.025000,315.500000,315.625000,319.498333,0.404671,318.237500,1.454630,316.539583,2.165680,316.129019,1.983907,-5.000000e-01,-8.660254e-01


(442, 20) (60, 20)


# 23. Strategy pattern for interchangeable forecasters

Both tree models use the same feature matrix and evaluation pipeline.

We use a small Strategy abstraction so model-specific fitting code does not leak into the evaluation logic.

In [65]:
class ForecastStrategy(ABC):
    @abstractmethod
    def fit(self, X, y):
        ...

    @abstractmethod
    def predict(self, X):
        ...

    @property
    @abstractmethod
    def name(self):
        ...


class SklearnStrategy(ForecastStrategy):
    def __init__(self, model, name):
        self.model = model
        self._name = name

    @property
    def name(self):
        return self._name

    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    def predict(self, X):
        return self.model.predict(X)

In [64]:
strategies = [
    SklearnStrategy(
        RandomForestRegressor(
            n_estimators=350 if not FAST_MODE else 160,
            min_samples_leaf=2,
            max_features=0.8,
            n_jobs=-1,
            random_state=SEED,
        ),
        "Random Forest",
    ),
    SklearnStrategy(
        HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=350 if not FAST_MODE else 180,
            max_leaf_nodes=21,
            l2_regularization=0.2,
            random_state=SEED,
        ),
        "Histogram Gradient Boosting",
    ),
]

ml_predictions = {}
rows = []

for strategy in strategies:
    strategy.fit(X_train, y_train)
    pred = strategy.predict(X_test)
    ml_predictions[strategy.name] = pd.Series(pred, index=y_test.index)
    rows.append(metric_table(y_test, pred, strategy.name))

display(pd.DataFrame(rows).sort_values("RMSE").round(4))

,model,MAE,RMSE
0,Random Forest,3.5854,4.3424
1,Histogram Gradient Boosting,4.7877,5.4497


In [66]:
show(time_plot(
    {
        "Actual": y_test,
        **ml_predictions,
    },
    "Tree-based lag-feature forecasting",
    y_label="CO2",
))

# 24. Random Forest versus gradient boosting

### Random Forest

Fits many decorrelated trees and averages them.

Strengths:

- robust;
- nonlinear;
- interaction-friendly;
- relatively low tuning sensitivity.

Weakness:

- interpolation is natural, extrapolation is poor.

### Gradient boosting

Fits trees sequentially to correct residual errors.

Strengths:

- often very strong tabular forecasting;
- good nonlinear interactions;
- efficient with lag/calendar features.

Weaknesses:

- sensitive to feature leakage;
- recursive multi-step forecasting can accumulate error;
- does not inherently encode temporal causal structure.

For both models:

> **Feature construction and evaluation protocol are part of the model.**

# Part XIII — Deep sequence models

# 25. A shared supervised sequence dataset

For neural sequence models we feed a fixed context window:

$$
[y_{t-L},\ldots,y_{t-1}]
\rightarrow
y_t.
$$

The architecture decides how information inside the window is combined.

We standardise using **training-only statistics** to avoid leakage.

In [67]:
series = co2.astype(float)
lookback = 24
test_n = 60

train_values = series.iloc[:-test_n].values
test_values = series.iloc[-(test_n+lookback):].values

mean_train = train_values.mean()
std_train = train_values.std()

scaled = (series.values - mean_train) / std_train

def make_sequences(values, lookback):
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i-lookback:i])
        y.append(values[i])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)

X_all, y_all = make_sequences(scaled, lookback)

split = len(X_all) - test_n
Xtr_seq, Xte_seq = X_all[:split], X_all[split:]
ytr_seq, yte_seq = y_all[:split], y_all[split:]

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(Xtr_seq).unsqueeze(-1),
        torch.tensor(ytr_seq).unsqueeze(-1),
    ),
    batch_size=32,
    shuffle=True,
)

Xte_t = torch.tensor(Xte_seq).unsqueeze(-1)

print(Xtr_seq.shape, Xte_seq.shape)

(442, 24) (60, 24)


In [68]:
def train_torch_model(model, train_loader, epochs=20, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    history = []
    model.train()

    for epoch in range(epochs):
        batch_losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        history.append(np.mean(batch_losses))

    return history


def torch_predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).squeeze(-1).cpu().numpy()


def unscale(v):
    return np.asarray(v) * std_train + mean_train


deep_actual = unscale(yte_seq)
deep_index = series.index[-test_n:]

# 26. Recurrent neural network: LSTM

A recurrent model processes the sequence step by step.

The hidden state acts as a learned summary of the past.

A vanilla RNN can suffer from vanishing/exploding gradients.

LSTM introduces gates that regulate:

- what to remember;
- what to forget;
- what to expose.

This improves long-range dependency learning.

In [69]:
class LSTMForecaster(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


lstm = LSTMForecaster(hidden_size=24)
lstm_history = train_torch_model(
    lstm,
    train_loader,
    epochs=10 if FAST_MODE else 40,
    lr=2e-3,
)

lstm_pred = unscale(torch_predict(lstm, Xte_t))

display(pd.DataFrame([metric_table(deep_actual, lstm_pred, "LSTM")]).round(4))

,model,MAE,RMSE
0,LSTM,5.7898,6.2834


In [70]:
show(time_plot(
    {
        "Actual": pd.Series(deep_actual, index=deep_index),
        "LSTM": pd.Series(lstm_pred, index=deep_index),
    },
    "LSTM one-step forecasting",
    y_label="CO2",
))

# Part XIV — Temporal Convolutional Networks

# 27. Why convolutions for sequences?

A Temporal Convolutional Network uses 1-D convolutions over time.

Key ideas:

- **causal convolution**: no future information;
- **dilation**: skip increasingly large gaps;
- **residual blocks**: stabilise deep training.

Dilated receptive fields grow quickly.

For kernel size $k=3$ and dilations $1,2,4,8$:

$$
\text{receptive field}
=
1+(k-1)(1+2+4+8).
$$

TCNs can process many time steps in parallel, unlike strictly recurrent architectures.

In [71]:
class CausalConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.padding = padding
        self.conv = nn.Conv1d(
            in_ch, out_ch,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
        )
        self.act = nn.ReLU()

    def forward(self, x):
        y = self.conv(x)
        if self.padding > 0:
            y = y[:, :, :-self.padding]
        return self.act(y)


class TCNForecaster(nn.Module):
    def __init__(self, channels=24):
        super().__init__()
        self.net = nn.Sequential(
            CausalConvBlock(1, channels, 3, 1),
            CausalConvBlock(channels, channels, 3, 2),
            CausalConvBlock(channels, channels, 3, 4),
        )
        self.head = nn.Linear(channels, 1)

    def forward(self, x):
        # [B,T,1] -> [B,1,T]
        z = x.transpose(1,2)
        z = self.net(z)
        return self.head(z[:, :, -1])


tcn = TCNForecaster(channels=20)
tcn_history = train_torch_model(
    tcn,
    train_loader,
    epochs=10 if FAST_MODE else 40,
    lr=2e-3,
)

tcn_pred = unscale(torch_predict(tcn, Xte_t))
display(pd.DataFrame([metric_table(deep_actual, tcn_pred, "TCN")]).round(4))

,model,MAE,RMSE
0,TCN,1.6773,1.9706


In [72]:
show(time_plot(
    {
        "Actual": pd.Series(deep_actual, index=deep_index),
        "TCN": pd.Series(tcn_pred, index=deep_index),
    },
    "Temporal Convolutional Network forecast",
    y_label="CO2",
))

# Part XV — Transformers for time series

# 28. Attention instead of recurrence

Self-attention compares every time position with every other position.

For query, key and value matrices:

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}
\right)V.
$$

This allows long-range relationships to be modelled directly rather than propagated through many recurrent steps.

However, vanilla attention has:

$$
O(L^2)
$$

memory/time with sequence length $L$.

This motivates sparse, patch-based and efficient-attention architectures in modern forecasting systems.

In [73]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerForecaster(nn.Module):
    def __init__(self, d_model=24, nhead=4, num_layers=2):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        self.pos = PositionalEncoding(d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=64,
            dropout=0.05,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        z = self.pos(self.embed(x))
        z = self.encoder(z)
        return self.head(z[:, -1, :])


transformer = TransformerForecaster(d_model=24, nhead=4, num_layers=1 if FAST_MODE else 2)
transformer_history = train_torch_model(
    transformer,
    train_loader,
    epochs=10 if FAST_MODE else 35,
    lr=1e-3,
)

transformer_pred = unscale(torch_predict(transformer, Xte_t))
display(pd.DataFrame([metric_table(deep_actual, transformer_pred, "Transformer")]).round(4))

,model,MAE,RMSE
0,Transformer,6.4758,6.931


In [74]:
show(time_plot(
    {
        "Actual": pd.Series(deep_actual, index=deep_index),
        "Transformer": pd.Series(transformer_pred, index=deep_index),
    },
    "Compact Transformer time-series forecaster",
    y_label="CO2",
))

# 29. Deep-learning comparison

One important lesson is that more sophisticated architecture does **not** guarantee better forecasting.

On a modest univariate dataset:

- strong seasonality;
- small sample size;
- smooth dynamics;

classical or tree-based models can outperform neural networks.

Deep models become more attractive when we have:

- many related series;
- large training corpora;
- high-dimensional covariates;
- long contexts;
- nonlinear interactions;
- representation transfer.

In [75]:
deep_rows = [
    metric_table(deep_actual, lstm_pred, "LSTM"),
    metric_table(deep_actual, tcn_pred, "TCN"),
    metric_table(deep_actual, transformer_pred, "Transformer"),
]

display(pd.DataFrame(deep_rows).sort_values("RMSE").round(4))

,model,MAE,RMSE
1,TCN,1.6773,1.9706
0,LSTM,5.7898,6.2834
2,Transformer,6.4758,6.9310


# Part XVI — Modern time-series foundation models

# 30. From task-specific models to pretrained forecasters

Traditional workflow:

$$
\text{dataset}
\rightarrow
\text{train model from scratch}
\rightarrow
\text{forecast}.
$$

Foundation-model workflow:

$$
\text{large pretraining corpus of many time series}
\rightarrow
\text{general pretrained model}
\rightarrow
\text{zero-shot/few-shot forecast}.
$$

Representative modern families include architectures based on:

- patch tokenisation;
- Transformer encoders/decoders;
- probabilistic tokenisation of numeric values;
- mixture training across domains and frequencies.

The key shift is **transfer learning across time series**.

# 31. Zero-shot forecasting intuition

A foundation model may receive only historical context:

$$
y_{t-L+1:t}
$$

and generate a future trajectory:

$$
\hat y_{t+1:t+H}
$$

without fitting model-specific parameters on the target series.

Potential advantages:

- strong cold-start behaviour;
- one model for many domains;
- probabilistic sample generation;
- less bespoke feature engineering.

Potential weaknesses:

- computational cost;
- distribution mismatch;
- opaque failure modes;
- reproducibility/versioning concerns;
- sometimes weaker performance than specialised local models;
- evaluation remains essential.

# 32. Optional foundation-model integration pattern

This environment does not currently have a dedicated time-series foundation-model package installed.

The following cell is intentionally **guarded**: it reports availability and shows the integration boundary without breaking the notebook.

Typical external libraries/models evolve quickly, so verify their current APIs before production use.

In [76]:
import importlib.util

foundation_packages = {
    "transformers": importlib.util.find_spec("transformers") is not None,
    "sktime": importlib.util.find_spec("sktime") is not None,
    "statsforecast": importlib.util.find_spec("statsforecast") is not None,
}

display(pd.Series(foundation_packages, name="installed"))

if not any(foundation_packages.values()):
    print(
        "No foundation-model forecasting stack is installed in this runtime. "
        "Keep the section conceptual unless you explicitly install and pin a supported library/model."
    )

transformers      True
sktime           False
statsforecast    False
Name: installed, dtype: bool

# Part XVII — Unified comparison of modelling paradigms

# 33. What each family assumes

| Family | Main object modelled | Strength | Typical weakness |
|---|---|---|---|
| ARIMAX | target + exogenous predictors + serial errors | interpretable external effects | future exog needed |
| Intervention | known event effect | explicit event modelling | causal assumptions |
| State-space | latent evolving states | missing data, filtering, components | model specification |
| GARCH | conditional variance recursion | financial volatility | limited latent flexibility |
| VAR | interacting stationary variables | system dynamics | parameter explosion |
| VECM | short-run changes + long-run equilibrium | cointegrated systems | rank/specification sensitivity |
| Hierarchical | aggregation constraints | coherent business forecasts | reconciliation covariance estimation |
| Conformal | empirical predictive errors | distribution-light intervals | dependence complicates guarantees |
| Tree ensembles | engineered lag/covariate features | strong nonlinear tabular model | weak extrapolation |
| RNN/LSTM | recurrent hidden state | sequence representation | sequential training |
| TCN | causal/dilated convolutions | parallelism, stable gradients | receptive-field design |
| Transformer | attention over context | long-range/global dependencies | quadratic vanilla attention |
| Foundation model | pretrained cross-series representation | zero-shot transfer | compute + distribution mismatch |

# 34. Complexity intuition

Let:

- $T$ = number of observations;
- $p$ = number of features;
- $L$ = sequence length;
- $d$ = hidden size;
- $N$ = number of trees.

Very rough computational perspectives:

### Linear AR/VAR-style estimation

Often dominated by linear algebra on lagged design matrices.

VAR parameter count grows roughly as:

$$
O(k^2p)
$$

for $k$ variables and lag order $p$.

### Random Forest

Training is approximately proportional to the work across $N$ trees and candidate splits.

### RNN

Sequential dependence roughly scales as:

$$
O(Ld^2)
$$

per sequence layer.

### TCN

Convolutions can parallelise over time and scale roughly linearly with $L$ for fixed kernel/channel sizes.

### Vanilla self-attention

Attention matrix construction is:

$$
O(L^2d).
$$

This $L^2$ term is why long-context forecasting architectures often use patching, sparse attention or decomposition.

# 35. Production decision guide

Use **ARIMAX** when:
- external covariates are interpretable and important;
- uncertainty and coefficients matter.

Use **intervention analysis** when:
- a known event occurred;
- estimating its immediate/persistent impact matters.

Use **state-space/Kalman** when:
- the underlying signal is latent;
- you need online filtering;
- observations are noisy/missing.

Use **GARCH** when:
- volatility clustering itself is the target.

Use **VAR/VECM** when:
- multiple series interact dynamically;
- long-run equilibrium relationships matter.

Use **hierarchical reconciliation** when:
- forecasts must add up across organisation/product/geography levels.

Use **conformal prediction** when:
- empirical interval calibration is important;
- distributional assumptions are uncertain.

Use **trees** when:
- lag/covariate feature engineering is feasible;
- nonlinear tabular interactions dominate.

Use **RNN/TCN/Transformers** when:
- large datasets or many related sequences exist;
- representation learning is valuable.

Use **foundation models** when:
- zero-shot/few-shot transfer is useful;
- you have rigorous benchmark infrastructure to verify that the pretrained model actually helps.

# Part XVIII — Exercises

## Exercise 1 — ARIMAX misspecification

Simulate a target driven by an exogenous variable plus AR(2) residuals.

Fit:

- OLS;
- ARIMAX(1,0,0);
- ARIMAX(2,0,0).

Compare residual Ljung–Box statistics and test RMSE.

---

## Exercise 2 — Pulse versus step intervention

Create:

- one temporary shock;
- one permanent level shift.

Fit the wrong intervention type intentionally.

Explain how coefficient interpretation changes.

---

## Exercise 3 — Structural break

Modify the structural-break simulation so only the slope changes.

Can you distinguish:

$$
\text{level break}
$$

from

$$
\text{trend break}?
$$

---

## Exercise 4 — Kalman filtering

Increase observation noise in the Nile/state-space example.

What happens to the smoothed latent level?

Explain using the Kalman gain.

---

## Exercise 5 — GARCH persistence

Simulate:

$$
(\alpha,\beta)
=
(0.05,0.7),
(0.08,0.90),
(0.10,0.885).
$$

Compare how long volatility shocks persist.

---

## Exercise 6 — VAR lag order

Fit VAR models with lags 1–8.

Compare:

- AIC;
- BIC;
- out-of-sample RMSE.

Does the information-criterion winner always forecast best?

---

## Exercise 7 — Cointegration failure

Simulate two independent random walks.

Run the Engle–Granger test.

Repeat many times and estimate the false-positive rate.

---

## Exercise 8 — Hierarchical reconciliation

Add a third region and two product categories.

Construct the summing matrix $S$ programmatically.

Verify numerically that reconciled forecasts satisfy:

$$
\tilde{\mathbf y}=S\tilde{\mathbf b}.
$$

---

## Exercise 9 — Conformal adaptation

Replace one fixed calibration quantile with a rolling 36-month quantile.

Introduce a variance regime change.

Which interval adapts faster?

---

## Exercise 10 — ML leakage

Create a deliberately leaked feature:

$$
X_t=Y_{t+1}.
$$

Observe the apparently excellent test metrics.

Explain why the result is invalid.

---

## Exercise 11 — Deep models

Increase lookback from 24 to 48 and 96.

Compare:

- LSTM;
- TCN;
- Transformer.

Does longer context always improve forecasting?

---

## Exercise 12 — Foundation-model benchmark

If you install a current pretrained forecasting package, compare it against:

- seasonal naïve;
- ETS/SARIMA;
- gradient boosting;
- your best neural model.

Use exactly the same rolling-origin evaluation windows.

# 36. Final synthesis

Advanced time-series analysis is not a single linear sequence of increasingly complicated models.

It is better understood as choosing the right **structure** for the problem.

### External drivers

$$
Y_t=f(X_t)+\text{temporal error}
$$

→ dynamic regression / ARIMAX.

### Latent states

$$
\text{hidden process}
\rightarrow
\text{noisy observation}
$$

→ state-space / Kalman filtering.

### Changing risk

$$
E[r_t^2\mid\mathcal F_{t-1}]
$$

→ ARCH/GARCH or stochastic volatility.

### Interacting variables

$$
\mathbf y_t
\leftrightarrow
\mathbf y_{t-1}
$$

→ VAR/VECM.

### Aggregation constraints

$$
\text{Total}=\sum\text{children}
$$

→ hierarchical reconciliation.

### Predictive uncertainty

$$
P(Y_{t+h}\in I_{t+h})
$$

→ probabilistic scoring and conformal methods.

### Flexible nonlinear prediction

$$
Y_t=f(\text{lags},\text{calendar},\text{covariates})
$$

→ tree ensembles.

### Representation learning

$$
\text{context sequence}
\rightarrow
\text{learned temporal representation}
\rightarrow
\text{forecast}
$$

→ RNNs, TCNs and Transformers.

### Cross-series transfer

$$
\text{pretraining over many datasets}
\rightarrow
\text{zero/few-shot forecasting}
$$

→ time-series foundation models.

The durable skill is therefore not asking:

> Which forecasting algorithm is best?

It is asking:

> **What kind of temporal structure, uncertainty, interaction, hierarchy, regime change and data scale does this problem actually contain—and which modelling family represents those properties most honestly?**